## Manipulation de données avec Python et Numpy

    Thomas Moreau : thomas.moreau@inria.fr

L'objectif de ce notebook est la prise en main de Python et Numpy en manipulant un célèbre jeu de données de machine learning.

Les données sont fournies par la librarie Python Scikit-Learn.

Vous travaillerez sur des données de chiffres manuscrits pour la classification (données `digits`)

## Questions de bases sur `numpy` 


**Exercice 1 :** (*Quelques rappels*)  
* Générer un tableau de 100 `0`.
* Remplacer la valeur 42 par `1`.
* Remplacer les 10 premières valeurs par `27`.
* Remplacer les 10 dernières valeurs par `-4`.
* Remplacer une valeur sur 2 par le double.
* Utilisé la méthode `reshape` pour obtenir un tableau de taille `5x20`.
* Multiplier une colonne sur 3 par `-1`.
* Retrancher `-3` aux valeurs non-nulles.
* Calculer la somme des valeurs dans le tableau.

### Opérations sur les tableaux

#### Opérations terme-à-terme

**Exercice 2 :** (*Courbe paramétrique*)  
Tracer la courbe $x = cos(t)$ et $y=sin(t)$ pour $t \in [0, 2\pi]$.  
On pourra utiliser la constante $\pi=$`np.pi`.

#### Broadcasting
Que donnerait les deux cas suivant?

```
Image  (3d array): 256 x 256 x 3
Scale  (1d array):             3
Result (3d array): 

A      (4d array):  8 x 1 x 6 x 1
B      (3d array):      7 x 1 x 5
Result (4d array):  
```

**Exercice 3 :** (*Opération terme-à-terme*)

Sans utiliser de boucles (`for/while`) :

 * Créer une matrice (5x6) aléatoire
 * Remplacer une colonne sur deux par sa valeur moins le double de la colonne suivante
 * Remplacer les valeurs négatives par 0 en utilisant un masque binaire


**Exercice 4 :** (*Résolution d'un système linéaire*)  
Résoudre le systeme d'équation: $\begin{cases}3x -2y +z &= 10\\x +5y + 10z &= 21\\y - z &= -5\\\end{cases}$.  
(*Astuce:* utiliser la fonction `np.linalg.inv`).

**Exercice 5 :** (*Blanchiment de donnée*)  
Créer un tableau X de taille `100x5` selon une loi normale ($\sim$ `np.random.randn`).  
Soustrayer à chaque colone sa moyenne et la diviser par son écart type.

# I - Manipulation et visualisation des *digits*

## Imports et intialisation

In [ ]:
%matplotlib inline                      

import numpy as np                      # charge un package pour le numérique
import matplotlib.pyplot as plt         # charge un package pour les graphiques

## Description du jeu de données

On charge le jeu de données *digits* disponible dans le package scikit-learn (nom d'import sklearn). Ce jeu de données contient des images de chiffres numérisés.

In [ ]:
# Chargement des données disponibles dans le package sklearn
from sklearn.datasets import load_digits

digits = load_digits()
X, y = digits.data, digits.target

In [ ]:
X.shape, y.shape

In [ ]:
print(f"Nombre de pixels :      {X.shape[1]}")
print(f"Nombre d'observations : {X.shape[0]}")
print(f"Nombre de classes :     {len(np.unique(y))}")

In [ ]:
# Choix d'une observation quelconque de la base
idx_to_test = 15

print("Affichage d'une ligne de la matrice / image:")
print(X[idx_to_test, :])
print("Affichage de la classe / chiffre associé:")
print(y[idx_to_test])

<div class="alert alert-success">
    <b>QUESTIONS:</b>
     <ul>
      <li>Quel est le dtype de X? y?</li>
      <li>Faites varier de `idx_to_test`. Sans afficher y[idx_to_test] arrivez-vous à reconnaitre le chiffre représenté?</li>
    </ul>
</div>

## Visualisation des observations:

Les images scannées sont de taille  8 x 8 et comportent donc 64 pixels chacune. Elles sont stockées sous la forme de vecteurs ligne, qu'il faut remettre dans un ordre lisible pour les identifier. L'affichage graphique est proposé avec les commandes qui suivent. On utilise la commande `np.reshape` pour passer d'un array 1d à un array 2d de 8x8 valeurs.

In [ ]:
# Utilisation de la fonction imshow pour l'affichage de l'image numéro idx_to_test:
plt.imshow(np.reshape(X[idx_to_test, :], (8, 8)));
plt.colorbar()

In [ ]:
np.unique(y)

In [ ]:
# Amélioration de la visualisation (niveau de gris) et de la légende:
plt.imshow(np.reshape(X[idx_to_test, :], (8, 8)),
           cmap='gray', aspect='equal', interpolation='nearest')

# Attention aux accents: ne pas oublier le u (Unicode) ci-dessous
plt.title(f"Le chiffre d'indice {idx_to_test} est un {y[idx_to_test]}"));

<div class="alert alert-success">
    <b>QUESTIONS:</b>
     <ul>
      <li>Afficher l'image précédente avec une ligne sur 2 et une colonne sur 2</li>
      <li>Afficher l'image précédente après avoir enlevé un pixel de chaque bord?</li>
      <li>Afficher l'histogramme des valeurs des pixels (fonction `plt.hist`)</li>
    </ul>
</div>


## Statistiques élementaires :
Pour mieux comprendre la base de données on va s'intéresser à quelques statistiques. 
On commence par calculer les moyennes et variances par classes pour chacun des chiffres. La moyenne par classe se visualise comme une image qui est une représentantion moyenne pour chaque chiffre de zéro à neuf. Idem pour la variance, ce qui permet alors de voir les parties avec les plus grandes variations entre les membres d'une même classe.


In [ ]:
classes_list = np.unique(y).astype(int)
print("Liste des classes en présence: ", classes_list)

<div class="alert alert-success">
    <b>QUESTIONS:</b>
     <ul>
      <li>Calculer un représentant moyen du chiffre 0 (l'image qui en pixel i,j contient la valeur moyenne du pixel i,j parmis tous les 0)</li>
      <li>Avec une boucle `for` calculer le représentant moyen pour chaque chiffre</li>
      <li>Faire la même chose en remplaçant la moyenne par l'écart type</li>
      <li>Afficher toutes les images sur une grille à l'aide de la fonction `plt.subplots`</li>
    </ul>
</div>


# II - Classification par plus proche centroide

Le but de cette partie est de vous faire implémenter votre propre classifieur
basé sur une idée très simple. Pour un nouveau chiffre, on prédit la classe
dont le chiffre moyen est le plus proche.

<div class="alert alert-success">
    <b>QUESTIONS:</b>
     <ul>
      <li>Partager la base de données en 2. On notera la première partie X_train, y_train et la deuxième partie X_test et y_test.</li>
      <li>Pour chaque chiffre calculer sur l'ensemble de train le chiffre moyen. On notera la variable contenant les moyennes `centroids_train`</li>
      <li>Pour chaque chiffre de l'ensemble de test, calculer le centroide le plus proche. Vous évaluerez si le chiffre ainsi obtenu correspond au vrai chiffre et en déduirez une estimation du pourcentage de bonnes prédictions.</li>
    </ul>
</div>

In [ ]:
n_samples, n_features = X.shape

n_samples_train = n_samples // 2

# Etape 1: Couper les données en 2 : train et test

X_train = X[:n_samples_train]
y_train = y[:n_samples_train]

X_test = X[n_samples_train:]
y_test = y[n_samples_train:]


<div class="alert alert-info">

**Bonus:**

Regrouper tout ce qu'on a fait auparavent dans des fonctions:

1. Écrire une fonction `split_train_test` qui prenne en entrée `(X, y)` et retourne 2 ensemble de données `(X_train, y_train)` et `(X_test, y_test)`. On pourra ajouter un paramètre `ratio` (float in [0, 1]) qui donne le rapport de proportion train/test.

2. Écrire une fonction `fit` qui prenne en entrée `X_train, y_train` et qui renvoie les centroids de chacune des classes `centroids`.

3. Écrire une fonction `predict` qui prenne en entrée `X_test` et `centroids` et qui renvoie en sortie la prédiction `y_pred` du modèle.

4. Faire une fonction `score` qui prend en entrée un couple `y` et `y_pred` et qui retourne l'accuracy pour ce couple.

</div>